In [ ]:
%%writefile matmul.cu

// C = A * B for square N x N matrices: serial, threaded, CUDA naive, CUDA tiled.

#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <chrono>
#include <thread>
#include <vector>
#include <algorithm>
#include <cuda_runtime.h>

#define TILE 16
#define NUM_THREADS 4

#define CHECK(call) do { \
    cudaError_t e = (call); \
    if (e != cudaSuccess) { printf("CUDA error: %s (line %d)\n", cudaGetErrorString(e), __LINE__); exit(1); } \
} while (0)

void matmulRows(const float* A, const float* B, float* C, int N, int rowStart, int rowEnd) {
    for (int i = rowStart; i < rowEnd; i++) {
        for (int j = 0; j < N; j++) C[i * N + j] = 0;
        for (int k = 0; k < N; k++) {
            float a = A[i * N + k];
            for (int j = 0; j < N; j++)
                C[i * N + j] += a * B[k * N + j];
        }
    }
}

void matmulThreaded(const float* A, const float* B, float* C, int N) {
    std::vector<std::thread> threads;
    int rowsPerThread = (N + NUM_THREADS - 1) / NUM_THREADS;
    for (int t = 0; t < NUM_THREADS; t++) {
        int start = t * rowsPerThread;
        int end = std::min(start + rowsPerThread, N);
        threads.emplace_back(matmulRows, A, B, C, N, start, end);
    }
    for (auto& th : threads) th.join();
}

// one thread per output element, everything read from global memory
__global__ void matmulNaive(const float* A, const float* B, float* C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row >= N || col >= N) return;

    float sum = 0;
    for (int k = 0; k < N; k++)
        sum += A[row * N + k] * B[k * N + col];
    C[row * N + col] = sum;
}

// same thing, but each block stages TILE x TILE chunks in shared memory
__global__ void matmulTiled(const float* A, const float* B, float* C, int N) {
    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];

    int tx = threadIdx.x, ty = threadIdx.y;
    int row = blockIdx.y * TILE + ty;
    int col = blockIdx.x * TILE + tx;
    float sum = 0;

    for (int t = 0; t < (N + TILE - 1) / TILE; t++) {
        int aCol = t * TILE + tx;
        int bRow = t * TILE + ty;
        As[ty][tx] = (row < N && aCol < N) ? A[row * N + aCol] : 0.0f;
        Bs[ty][tx] = (bRow < N && col < N) ? B[bRow * N + col] : 0.0f;
        __syncthreads();    // tile must be fully loaded before anyone reads it

        for (int k = 0; k < TILE; k++)
            sum += As[ty][k] * Bs[k][tx];
        __syncthreads();    // and fully used before the next iteration overwrites it
    }

    if (row < N && col < N)
        C[row * N + col] = sum;
}

float maxDiff(const float* X, const float* Y, size_t n) {
    float m = 0;
    for (size_t i = 0; i < n; i++) m = fmaxf(m, fabsf(X[i] - Y[i]));
    return m;
}

template <typename F> double cpuTime(F work) {
    auto t0 = std::chrono::high_resolution_clock::now();
    work();
    std::chrono::duration<double, std::milli> d = std::chrono::high_resolution_clock::now() - t0;
    return d.count();
}

template <typename F> float gpuTime(F launch) {
    cudaEvent_t a, b;
    CHECK(cudaEventCreate(&a));
    CHECK(cudaEventCreate(&b));
    CHECK(cudaEventRecord(a));
    launch();
    CHECK(cudaEventRecord(b));
    CHECK(cudaEventSynchronize(b));    // launches are async
    float ms;
    CHECK(cudaEventElapsedTime(&ms, a, b));
    cudaEventDestroy(a);
    cudaEventDestroy(b);
    return ms;
}

int main(int argc, char** argv) {
    int N = (argc > 1) ? atoi(argv[1]) : 1024;
    size_t count = (size_t)N * N;
    size_t bytes = count * sizeof(float);
    printf("Matrix size: %d x %d\n\n", N, N);

    std::vector<float> A(count), B(count);
    std::vector<float> Cserial(count), Cthread(count), Cnaive(count), Ctiled(count);
    srand(42);
    for (size_t i = 0; i < count; i++) {
        A[i] = (float)rand() / RAND_MAX;
        B[i] = (float)rand() / RAND_MAX;
    }

    double serialMs = cpuTime([&] { matmulRows(A.data(), B.data(), Cserial.data(), N, 0, N); });
    double threadMs = cpuTime([&] { matmulThreaded(A.data(), B.data(), Cthread.data(), N); });

    float *dA, *dB, *dC;
    CHECK(cudaMalloc(&dA, bytes));
    CHECK(cudaMalloc(&dB, bytes));
    CHECK(cudaMalloc(&dC, bytes));
    CHECK(cudaMemcpy(dA, A.data(), bytes, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(dB, B.data(), bytes, cudaMemcpyHostToDevice));

    dim3 block(TILE, TILE);
    dim3 grid((N + TILE - 1) / TILE, (N + TILE - 1) / TILE);
    auto runNaive = [&] { matmulNaive<<<grid, block>>>(dA, dB, dC, N); };
    auto runTiled = [&] { matmulTiled<<<grid, block>>>(dA, dB, dC, N); };

    runNaive();    // warm-up, first launch is slow
    CHECK(cudaDeviceSynchronize());

    float naiveMs = gpuTime(runNaive);
    CHECK(cudaMemcpy(Cnaive.data(), dC, bytes, cudaMemcpyDeviceToHost));

    CHECK(cudaMemset(dC, 0, bytes));    // so a silent kernel failure can't pass the check
    float tiledMs = gpuTime(runTiled);
    CHECK(cudaMemcpy(Ctiled.data(), dC, bytes, cudaMemcpyDeviceToHost));

    printf("%-22s %10s %10s %12s\n", "Version", "Time (ms)", "Speedup", "Max diff");
    printf("%-22s %10.2f %9.2fx %12s\n", "Serial CPU", serialMs, 1.0, "-");
    printf("%-22s %10.2f %9.2fx %12g\n", "Threaded CPU", threadMs, serialMs / threadMs, maxDiff(Cserial.data(), Cthread.data(), count));
    printf("%-22s %10.2f %9.2fx %12g\n", "CUDA naive", naiveMs, serialMs / naiveMs, maxDiff(Cserial.data(), Cnaive.data(), count));
    printf("%-22s %10.2f %9.2fx %12g\n", "CUDA tiled", tiledMs, serialMs / tiledMs, maxDiff(Cserial.data(), Ctiled.data(), count));

    cudaFree(dA); cudaFree(dB); cudaFree(dC);
    return 0;
}

Overwriting matmul.cu


In [ ]:
!nvcc -O2 -arch=sm_75 matmul.cu -o matmul
!./matmul 128
!./matmul 512
!./matmul 1024
!./matmul 2048

Matrix size: 128 x 128

Version                 Time (ms)    Speedup     Max diff
Serial CPU                   2.23      1.00x            -
Threaded CPU                 1.94      1.15x            0
CUDA naive                   0.03     84.75x  1.14441e-05
CUDA tiled                   0.03     70.33x  1.14441e-05
Matrix size: 512 x 512

Version                 Time (ms)    Speedup     Max diff
Serial CPU                 100.83      1.00x            -
Threaded CPU                97.08      1.04x            0
CUDA naive                   0.59    170.33x  4.57764e-05
CUDA tiled                   0.40    252.79x  4.57764e-05
Matrix size: 1024 x 1024

Version                 Time (ms)    Speedup     Max diff
Serial CPU                 750.49      1.00x            -
Threaded CPU               765.99      0.98x            0
CUDA naive                   4.28    175.42x  9.15527e-05
CUDA tiled                   2.69    278.86x  9.15527e-05
Matrix size: 2048 x 2048

Version                 Time (